# 在 Eino 中创建 Tool，主要有四种方式。
## 最推荐的是 utils.InferTool，它能直接从 Go 结构体的 tag 推导出参数 Schema，避免手动维护两份定义。

    方式一：使用 InferTool（推荐）
        这是最简洁的方式。你只需定义一个参数结构体，用 jsonschema tag 描述字段约束，InferTool 会自动生成 ToolInfo。

        ```go
        import (
            "context"
            "github.com/cloudwego/eino/components/tool"
            "github.com/cloudwego/eino/components/tool/utils"
        )

        type WeatherParams struct {
            City string `json:"city" jsonschema:"required,description=城市名称"`
            Unit string `json:"unit" jsonschema:"enum=celsius,enum=fahrenheit,description=温度单位"`
        }

        func getWeather(ctx context.Context, params *WeatherParams) (string, error) {
            return "晴，25°C", nil
        }

        // 创建工具
        weatherTool, err := utils.InferTool(
            "get_weather",
            "查询指定城市的天气",
            getWeather,
        )
        ```

    关键点：json tag 控制参数名，jsonschema tag 中的 required 标记必填，description 提供字段说明，enum 限定可选值。

    方式二：使用 NewTool（手动定义 Schema）
        当参数结构无法用 Go struct 表达（例如需要动态构建 Schema），或你希望完全手动控制时使用。

        ```go
        import "github.com/cloudwego/eino/schema"

        toolInfo := &schema.ToolInfo{
            Name: "get_weather",
            Desc: "查询指定城市的天气",
            ParamsOneOf: schema.NewParamsOneOfByParams(map[string]*schema.ParameterInfo{
                "city": {Type: "string", Desc: "城市名称", Required: true},
            }),
        }

        weatherTool := utils.NewTool(toolInfo, getWeather)
        ```
        缺点是 ToolInfo 中的参数定义与 getWeather 的参数结构是分开维护的，容易不一致。

        方式三：实现 Tool 接口
            适用于需要完全自定义逻辑的场景（如动态 Info、特殊序列化处理）。以 InvokableTool 为例：

            ```go
            type MyTool struct{}

            func (t *MyTool) Info(ctx context.Context) (*schema.ToolInfo, error) {
                return &schema.ToolInfo{ /* ... */ }, nil
            }

            func (t *MyTool) InvokableRun(ctx context.Context, argumentsInJSON string, opts ...tool.Option) (string, error) {
                // 需要自行反序列化 argumentsInJSON
                return "result", nil
            }
            ```
            这种方式最灵活，但需要手动处理 JSON 反序列化。

        方式四：使用官方封装工具
            Eino 生态（eino-ext）提供了开箱即用的工具，直接引入即可使用。例如 DuckDuckGo 搜索：

            ```go
            import "github.com/bytedance/eino-ext/components/tool/duckduckgo"

            searchTool, err := duckduckgo.NewTool(ctx, &duckduckgo.Config{ /* ... */ })
            ```

## 补充：多模态与流式工具
        如果你需要返回图片等多模态结果，可使用 utils.InferEnhancedTool；需要流式返回多模态内容，使用 utils.InferEnhancedStreamTool。它们与 InferTool 用法类似，只是函数签名不同。
        选型建议：优先用 InferTool；需要动态 Schema 或特殊控制时用 NewTool；有极端定制需求时实现接口；通用能力优先找官方封装。

# DeepAgent 介绍

    Eino 的 DeepAgent 是一个基于 ChatModelAgent 的开箱即用高级智能体，核心价值在于内置了任务规划、子代理委派、上下文隔离能力。它适合处理“需要先拆解步骤、再分派执行”的复杂多步任务，而简单的单工具调用场景用它反而会增加不必要的 token 开销。

    🧭 适用场景：什么时候用 DeepAgent
        根据官方对比，DeepAgent 与普通 ChatModelAgent 的选型边界很清晰：
        适合用 DeepAgent 的场景：
            1、任务需要显式规划（如“先分析数据、再写代码、最后生成报告”）
            2、需要文件系统操作（读写文件、glob 搜索、grep 查找）
            3、需要执行 Shell 命令或运行 Python 代码
            4、任务可以拆给多个专业子代理，且子代理之间不需要共享完整对话历史
            5、希望用内置 Prompt 和工具快速搭出一个“能干活”的 Agent，而不是从零拼装

        适合用普通 ChatModelAgent 的场景：
            1、任务简单，只需调用一个或少数几个固定工具
            2、你需要对 Prompt、工具绑定、上下文管理做细粒度控制
            3、对 token 成本敏感（DeepAgent 的规划与委派会带来额外模型调用）

    🏗️ 核心架构与内置工具
        DeepAgent 的 Config 结构中最关键的部分如下：
        ```go
        type Config struct {
            Name        string              // Agent 名称
            Description string              // 用途描述
            ChatModel   model.BaseModel     // 必填，需支持工具调用
            Instruction string              // 系统提示词，为空则用内置默认

            SubAgents   []adk.Agent         // 子代理列表，绑定到 task 工具
            ToolsConfig adk.ToolsConfig     // 自定义工具

            Backend        filesystem.Backend   // 注册 ls/read_file/write_file/edit_file/glob/grep
            Shell          filesystem.Shell      // 注册 execute（Shell 命令执行）

            WithoutWriteTodos      bool // 关闭内置规划工具
            WithoutGeneralSubAgent bool // 关闭默认 general-purpose 子代理
        }
        ```
        内置能力一览：
            1、write_todos：将复杂任务拆解为 TODO 列表并跟踪进度
            2、文件系统工具：配置 Backend 后自动获得 ls、read_file、write_file、edit_file、glob、grep
            3、Shell 执行：配置 Shell 后获得 execute 工具，可运行命令或脚本
            4、task 委派：主代理通过它调用子代理，子代理上下文隔离（只接收任务描述，不继承完整对话历史）
            5、general-purpose 子代理：默认启用，当没有专门子代理匹配时使用